In [1]:
import pandas as pd
import os
import pinecone
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

load_dotenv()

/opt/anaconda3/envs/langchain_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
files = pd.read_csv("course_section_descriptions.csv", encoding="cp1252")

In [5]:
files["unique_id"] = (
    files["course_id"].astype(str) + "-" + files["section_id"].astype(str)
)

In [7]:
files["metadata"] = files.apply(
    lambda row: {
        "course_name": row["course_name"],
        "section_name": row["section_name"],
        "section_description": row["section_description"],
    },
    axis=1,
)

In [8]:
files.head()

,course_id,course_name,course_slug,course_description,course_description_short,course_technology,course_topic,course_instructor_quote,section_id,section_name,section_description,unique_id,metadata
0,2,Introduction to Tableau,tableau,Tableau is now one of the most popular busines...,Teaching you how to tell compelling stories wi...,tableau,data visualization,Data scientists don’t just need to deal with d...,9,Introduction to Tableau,While Tableau is an indispensable tool in the ...,2-9,"{'course_name': 'Introduction to Tableau', 'se..."
1,2,Introduction to Tableau,tableau,Tableau is now one of the most popular busines...,Teaching you how to tell compelling stories wi...,tableau,data visualization,Data scientists don’t just need to deal with d...,10,Tableau Functionalities,"In this section, you will create your first Ta...",2-10,"{'course_name': 'Introduction to Tableau', 'se..."
2,2,Introduction to Tableau,tableau,Tableau is now one of the most popular busines...,Teaching you how to tell compelling stories wi...,tableau,data visualization,Data scientists don’t just need to deal with d...,11,The Tableau Exercise,This section is a practical example that will ...,2-11,"{'course_name': 'Introduction to Tableau', 'se..."
3,3,The Complete Data Visualization Course with Py...,data-visualization,The Data Visualization course is designed for ...,Teaching you how to master the art of creating...,python,data visualization,Data visualization is the face of data. Many p...,12,Introduction,"In this section, you will learn about the impo...",3-12,{'course_name': 'The Complete Data Visualizati...
4,3,The Complete Data Visualization Course with Py...,data-visualization,The Data Visualization course is designed for ...,Teaching you how to master the art of creating...,python,data visualization,Data visualization is the face of data. Many p...,13,Setting Up the Environments,"Here, we set up different environments for the...",3-13,{'course_name': 'The Complete Data Visualizati...


In [9]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [10]:
def create_embeddings(row):
    combined_text = " ".join(
        [
            str(row[field])
            for field in [
                "course_name",
                "course_technology",
                "course_description",
                "section_name",
                "section_description",
            ]
        ]
    )
    embedding = model.encode(combined_text, show_progress_bar=False).tolist()
    return embedding

In [11]:
files["embeddings"] = files.apply(create_embeddings, axis=1)
files.head()

,course_id,course_name,course_slug,course_description,course_description_short,course_technology,course_topic,course_instructor_quote,section_id,section_name,section_description,unique_id,metadata,embeddings
0,2,Introduction to Tableau,tableau,Tableau is now one of the most popular busines...,Teaching you how to tell compelling stories wi...,tableau,data visualization,Data scientists don’t just need to deal with d...,9,Introduction to Tableau,While Tableau is an indispensable tool in the ...,2-9,"{'course_name': 'Introduction to Tableau', 'se...","[0.0027814852073788643, -0.02600695565342903, ..."
1,2,Introduction to Tableau,tableau,Tableau is now one of the most popular busines...,Teaching you how to tell compelling stories wi...,tableau,data visualization,Data scientists don’t just need to deal with d...,10,Tableau Functionalities,"In this section, you will create your first Ta...",2-10,"{'course_name': 'Introduction to Tableau', 'se...","[0.01200291607528925, -0.017747530713677406, -..."
2,2,Introduction to Tableau,tableau,Tableau is now one of the most popular busines...,Teaching you how to tell compelling stories wi...,tableau,data visualization,Data scientists don’t just need to deal with d...,11,The Tableau Exercise,This section is a practical example that will ...,2-11,"{'course_name': 'Introduction to Tableau', 'se...","[0.020227862522006035, -0.030901920050382614, ..."
3,3,The Complete Data Visualization Course with Py...,data-visualization,The Data Visualization course is designed for ...,Teaching you how to master the art of creating...,python,data visualization,Data visualization is the face of data. Many p...,12,Introduction,"In this section, you will learn about the impo...",3-12,{'course_name': 'The Complete Data Visualizati...,"[0.07207506150007248, -0.02398512326180935, -0..."
4,3,The Complete Data Visualization Course with Py...,data-visualization,The Data Visualization course is designed for ...,Teaching you how to master the art of creating...,python,data visualization,Data visualization is the face of data. Many p...,13,Setting Up the Environments,"Here, we set up different environments for the...",3-13,{'course_name': 'The Complete Data Visualizati...,"[0.0787239521741867, -0.02722296305000782, -0...."


In [12]:
pc = Pinecone(
    api_key=os.getenv("PINECONE_API_KEY"), environment=os.getenv("PINECONE_ENVIRONMENT")
)

In [13]:
index_name = "my-index"
dimensions = 384
metric = "cosine"

In [14]:
if index_name in [index.name for index in pc.list_indexes()]:
    print(f"Index '{index_name}' already exists.")
    pc.delete_index(index_name)
    print(f"Index '{index_name}' deleted.")
else:
    print(f"Index '{index_name}' does not exist.")

Index 'my-index' already exists.
Index 'my-index' deleted.


In [15]:
pc.create_index(
    name=index_name,
    dimension=dimensions,
    metric=metric,
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)

{
    "name": "my-index",
    "metric": "cosine",
    "host": "my-index-h2xw5gc.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "access-control-allow-origin": "*",
            "vary": "origin,access-control-request-method,access-control-request-headers",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "2025-10",
  

In [16]:
index = pc.Index(index_name)

In [17]:
vectors_to_upsert = [
    (row["unique_id"], row["embeddings"], row["metadata"])
    for _, row in files.iterrows()
]

In [18]:
index.upsert(vectors=vectors_to_upsert)
print("Vectors upserted successfully.")

Vectors upserted successfully.


In [19]:
query = "clustering"
query_embedding = model.encode(query).tolist()

In [20]:
query_results = index.query(
    vector=[query_embedding], top_k=12, include_metadata=True, include_values=True
)

In [21]:
for match in query_results.matches:
    course_details = match.get("metadata", {})
    course_name = course_details.get("course_name", "N/A")
    section_name = course_details.get("section_name", "N/A")
    section_description = course_details.get("section_description", "N/A")

    print(f"Match ID: {match['id']}, Score: {match['score']:.4f}")
    print(f"Course Name: {course_name}")
    print(f"Section Name: {section_name}")
    print(f"Section Description: {section_description}")
    print()

Match ID: 51-469, Score: 0.5456
Course Name: Machine Learning in Excel
Section Name: Cluster Analysis
Section Description: Cluster analysis is the most intuitive and important example of unsupervised learning. However, to be able to understand cluster analysis, you must first become familiar with the mathematics behind it. Here we will explore the fundamentals of cluster analysis and have a look at the differences between clustering and classification.

Match ID: 37-374, Score: 0.5433
Course Name: Machine Learning in Python
Section Name: Other Types of Clustering
Section Description: In previous sections, we focus extensively on k-means clustering, as it is the fastest and most efficient method for clustering. In this section, we explore other approaches that are less common.

Match ID: 51-470, Score: 0.5081
Course Name: Machine Learning in Excel
Section Name: K-means Clustering
Section Description: Master K-means clustering in Excel by learning how to choose the number of clusters in 